In [1]:
# ─────────────────────────────────────────────────────────────────────
# NOTEBOOK 01 · CELL 1 · SETUP
# Purpose: load tools, define folders, and write down the cricket rules
#          we agreed on, so every later cell uses the SAME definitions.
# ─────────────────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")                 # keep the notebook output clean
pd.set_option("display.max_columns", 60)          # show wide tables fully
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ---- Folders -----------------------------------------------------------
DATA_DIR = Path(".")        # TECHNICAL: raw CSVs sit next to this notebook
OUT_DIR  = Path("interim")  # cleaned tables are saved here for notebook 02 onwards
OUT_DIR.mkdir(exist_ok=True)

SEED = 42                   # TECHNICAL: a fixed random seed makes every re-run give identical
np.random.seed(SEED)        #            results (reproducibility is part of the judging criteria)

# ---- Cricket rules (decided with you; kept in ONE place so the report can cite them) ----
# FUNCTIONAL: a T20 over has 6 *legal* balls. Wides and no-balls are re-bowled, so they do not
# use up one of the 120 balls an innings is allowed. "other" behaves like a no-ball variant
# (we proved this from the over structure; Cell 5 shows the evidence).
ILLEGAL_EXTRAS = ["wide", "Overstep", "other"]
MAX_LEGAL_BALLS = 120       # 20 overs x 6 balls: no innings may exceed this
# FUNCTIONAL: plausible bowling speeds (km/h). Anything outside is a sensor/entry error.
SPEED_RANGE = {"Pace": (100, 160), "Spin": (50, 125)}

# ---- Data-quality log -----------------------------------------------------
# TECHNICAL: every fix we make is recorded here, so the methodology report can say exactly
# what was changed, why, and how many rows it touched.
dq_log = []
def log_dq(step, issue, rows_affected, action):
    dq_log.append(dict(step=step, issue=issue, rows_affected=int(rows_affected), action=action))
    print(f"[DQ] {step} | {issue} | {int(rows_affected):,} rows | {action}")

print("Setup complete. pandas", pd.__version__)

Setup complete. pandas 1.4.3


### Cell 1: Setup
**Procedure:** Imported libraries, fixed the random seed (42), created the `interim/` output folder, and defined the cricket rules used everywhere downstream: illegal extras (`wide`, `Overstep`, `other`), 120 legal balls per innings, plausible speed ranges, and a data-quality log (`dq_log`) that records every fix with its row count.
**Observation:** No data touched yet. The rules are defined once, so every later notebook uses identical definitions.

In [2]:
# ─────────────────────────────────────────────────────────────────────
# CELL 2 · LOAD THE SIX TABLES (+ the age file)
# Purpose: bring everything into memory and take a first look at its size.
# ─────────────────────────────────────────────────────────────────────
# TECHNICAL GOTCHA: in ball_by_ball.csv the text "None" in `extras_type` means "no extra on this
# ball". By default pandas turns the word "None" into a missing value (NaN), which would hide the
# difference between "no extra" and "unknown". We keep "None" as a real word and only treat truly
# empty cells as missing.
# Use the organisers' corrected file; the old one has a category code where the extras runs should be.
ball = pd.read_csv(DATA_DIR / "updated_ball_by_ball.csv", keep_default_na=False, na_values=[""], low_memory=False)

matches = pd.read_csv("matches.csv")
comps   = pd.read_csv("competitions.csv")
teams   = pd.read_csv("teams.csv")
xi      = pd.read_csv("playing_eleven.csv")
squads  = pd.read_csv("competition_team_player_mapping.csv")
age     = pd.read_excel("Player_age_updated.xlsx", sheet_name="Sheet1")   # player_id, player_name, age

tables = {"ball_by_ball": ball, "matches": matches, "competitions": comps,
          "teams": teams, "playing_eleven": xi, "squads (comp-team-player)": squads, "age file": age}

summary = pd.DataFrame({
    name: {"rows": len(df), "columns": df.shape[1], "memory_MB": df.memory_usage(deep=True).sum() / 1e6}
    for name, df in tables.items()
}).T
display(summary)

# A first look at one delivery, rotated so every column is readable
display(ball.head(3).T)

,rows,columns,memory_MB
ball_by_ball,"200,999.000",33.000,196.795
matches,883.000,22.000,0.534
competitions,141.000,4.000,0.032
teams,353.000,3.000,0.030
playing_eleven,"28,381.000",9.000,3.221
squads (comp-team-player),"6,413.000",6.000,1.610
age file,"1,210.000",3.000,0.105


,0,1,2
match_id,13511,13511,13511
innings,1,1,1
overid,1,1,1
over_ball_num,1,2,3
innings_ball_num,1,2,3
batting_team_id,22,22,22
bowling_team_id,28,28,28
striker_id,1187,1187,1187
striker_name,Jason Roy,Jason Roy,Jason Roy
non_striker_id,6273,6273,6273


### Cell 2: Load the tables
**Procedure:** Loaded the six competition tables plus the age file. Read `ball_by_ball.csv` with `keep_default_na=False` so the text "None" in `extras_type` stays a real value (no extra) rather than becoming missing.

**Observations**
- 200,999 deliveries across 883 matches, 141 competitions, 353 teams, 28,381 playing-XI rows, 6,413 squad rows and 1,210 players with an age.
- Each row is one delivery, with striker, non-striker, bowler, runs, extras, wicket and shot details.
- Memory use is about 197 MB, so everything fits comfortably in RAM and no chunking is needed.
- The first deliveries show `bowl_speed = 0`. That is the corrupted field we handle later, not a real speed.

In [3]:
import pandas as pd, numpy as np

# TECHNICAL: load only the columns that matter; the key (match, innings, ball number) lines the files up.
cols = ["match_id","innings","innings_ball_num","runs","extras_type","extras1_runs","extras_2_runs"]
new = pd.read_csv("updated_ball_by_ball.csv", usecols=cols)

# FUNCTIONAL: two ways to count a team's runs on each ball.
# OLD rule (what we used): bat runs + 1 for any extra + overthrow runs.
new["old_total"] = new.runs + new.extras_type.notna().astype(int) + new.extras_2_runs
# NEW rule: bat runs + the real extras runs from the corrected file + overthrow runs.
new["new_total"] = new.runs + new.extras1_runs + new.extras_2_runs

diff = new[new.old_total != new.new_total]
print("balls where the two rules differ:", len(diff), "of", len(new))
print("runs gap in total:", int((new.new_total - new.old_total).sum()))
print(diff.groupby("extras_type")[["old_total","new_total"]].sum())

# Compare innings totals; a wide or no-ball worth more than 1 run would show up here.
inn = new.groupby(["match_id","innings"])[["old_total","new_total"]].sum()
print("innings where totals differ:", (inn.old_total != inn.new_total).sum(), "of", len(inn))

balls where the two rules differ: 190327 of 200999
runs gap in total: -186376
             old_total  new_total
extras_type                      
None            457452     268286
Penalty              1          5
byes               181        652
legbyes            373       1228
wide               607       2067
innings where totals differ: 1723 of 1724


In [4]:
# ─────────────────────────────────────────────────────────────────────
# CELL 3 · STRUCTURAL AUDIT: are the keys, joins and IDs trustworthy?
# Purpose: before any cricket logic, confirm the tables fit together.
# ─────────────────────────────────────────────────────────────────────
checks = []
def add(check, result, why_it_matters):
    checks.append({"check": check, "result": result, "why it matters": why_it_matters})

# 1) Is every delivery uniquely identified?  (match_id, innings, innings_ball_num) must be a key.
dup_balls = ball.duplicated(["match_id", "innings", "innings_ball_num"]).sum()
add("Duplicate deliveries (same match, innings, ball number)", dup_balls, "Duplicates would double-count runs/wickets")

# 2) Which matches have no ball-by-ball data, and is there a good reason?
no_ball = matches[~matches.match_id.isin(ball.match_id)]
all_abandoned = no_ball.result.str.contains("Abandon|No Result", case=False).all()
add("Matches in matches.csv", len(matches), "Universe of games")
add("Matches with ball-by-ball data", ball.match_id.nunique(), "What we can actually analyse")
add("Matches WITHOUT ball data", len(no_ball), "Should be washed-out games only")
add("...and all of them are abandoned / no result?", bool(all_abandoned), "If True, nothing real is missing")

# 3) What innings numbers exist?  (1, 2 = the match; 3, 4 = super overs)
add("Innings values present", sorted(ball.innings.unique().tolist()), "3 and 4 are super overs (excluded later)")

# 4) Player identity: can we trust IDs? Names can clash, IDs must not.
pl = pd.concat([ball[["striker_id", "striker_name"]].set_axis(["id", "name"], axis=1),
                ball[["bowler_id",  "bowler_name"]].set_axis(["id", "name"], axis=1),
                squads[["player_id", "player_name"]].set_axis(["id", "name"], axis=1)]).drop_duplicates()
ids_many_names = (pl.groupby("id").name.nunique() > 1).sum()
names_many_ids = pl.groupby("name").id.nunique()
names_many_ids = names_many_ids[names_many_ids > 1]
add("Player IDs that carry more than one name", ids_many_names, "Should be 0: IDs are reliable")
add("Names that map to more than one player ID", len(names_many_ids), "Real clashes: we must join on ID, never on name")

# 5) Age coverage for everyone who actually played
active_ids = set(ball.striker_id) | set(ball.bowler_id)
no_age = sorted(active_ids - set(age.player_id))
add("Players seen in ball data", len(active_ids), "")
add("...of whom have NO age in the age file", len(no_age), f"IDs: {no_age}")

audit = pd.DataFrame(checks)
display(audit)

print("\nNames shared by two different players:")
display(pl[pl.name.isin(names_many_ids.index)].sort_values("name").reset_index(drop=True))

,check,result,why it matters
0,"Duplicate deliveries (same match, innings, bal...",0,Duplicates would double-count runs/wickets
1,Matches in matches.csv,883,Universe of games
2,Matches with ball-by-ball data,867,What we can actually analyse
3,Matches WITHOUT ball data,16,Should be washed-out games only
4,...and all of them are abandoned / no result?,True,"If True, nothing real is missing"
5,Innings values present,"[1, 2, 3, 4]",3 and 4 are super overs (excluded later)
6,Player IDs that carry more than one name,1,Should be 0: IDs are reliable
7,Names that map to more than one player ID,7,"Real clashes: we must join on ID, never on name"
8,Players seen in ball data,958,
9,...of whom have NO age in the age file,5,"IDs: [6922, 20229, 28756, 29833, 39444]"



Names shared by two different players:


,id,name
0,7446,Abdul Samad
1,6922,Abdul Samad
2,29833,Bashir Ahmad
3,39479,Bashir Ahmad
4,8638,Basil Hameed
5,39444,Basil Hameed
6,7338,Haider Ali
7,28756,Haider Ali
8,20229,Mohammad Ali
9,3083,Mohammad Ali


### Cell 3: Structural audit
**Procedure:** Checked that the key (match, innings, ball number) is unique, that matches without ball data have a legitimate reason, that player IDs are reliable, and how well ages cover active players.

**Observations**
- 0 duplicate deliveries, so nothing is double-counted.
- 867 of 883 matches have ball data. The 16 without it are all abandoned or no-result games, so no real match is missing.
- Innings values are 1 to 4. Innings 3 and 4 are super overs, handled in Cell 4.
- Every player ID has exactly one name, so IDs are trustworthy. But 7 names belong to two different players (Rashid Khan 4722/20172, Abdul Samad 7446/6922, Haider Ali, Mohammad Ali and others). **Decision: join on `player_id` only, never on name.**
- 958 players appear in the ball data. 5 have no age (IDs 6922, 20229, 28756, 29833, 39444), so they will carry a "missing age" flag. They must not be silently dropped, because the RR age-fit logic needs to treat them explicitly.

In [5]:
# ─────────────────────────────────────────────────────────────────────
# CELL 4 · SCOPE: label every match, drop super overs, flag odd matches
# Purpose: decide exactly which deliveries are in the analysis.
# ─────────────────────────────────────────────────────────────────────
# TECHNICAL: RE-RUN SAFETY ("idempotent" cell). This cell renames columns, merges tables and drops
# rows. If you ran it twice on the same objects, the second run would fail (the old column name
# no longer exists) or create duplicate columns. So we start from FRESH copies of the raw tables
# every time. Rule of thumb: a cell that modifies data should be safe to run again.
matches = pd.read_csv("matches.csv")
comps   = pd.read_csv("competitions.csv")
# Use the organisers' corrected file; the old one has a category code where the extras runs should be.
ball = pd.read_csv(DATA_DIR / "updated_ball_by_ball.csv", keep_default_na=False, na_values=[""], low_memory=False)
dq_log  = [d for d in dq_log if d["step"] != "scope"]      # drop old 'scope' log lines from earlier runs

# 4a) FUNCTIONAL: give every competition a "tier". IPL and T20I are what the brief names; the two
# A-team / emerging tournaments are kept on purpose (RR scouts young talent) but are labelled
# separately so we can adjust for their different opposition level later.
comps["tier"] = np.select(
    [comps.comp_name.str.startswith("IPL"),
     comps.comp_name.str.contains("Emerging|Rising Stars", case=False)],
    ["IPL", "A-team / emerging"], default="T20I")

matches = matches.merge(comps[["comp_id", "comp_name", "match_format", "season", "tier"]], on="comp_id", how="left")
matches["year"] = pd.to_datetime(matches.match_date).dt.year

# 4b) TECHNICAL: the columns are called "..._revised_overs" but hold BALLS (114 = 19 overs).
# Proof: every value is a multiple of 6.  We rename them so nobody misreads them later.
for col in ["first_inn_revised_overs", "second_inn_revised_overs"]:
    vals = matches[col].dropna()
    assert (vals % 6 == 0).all(), f"{col} is not in balls"
matches = matches.rename(columns={"first_inn_revised_overs": "first_inn_revised_balls",
                                  "second_inn_revised_overs": "second_inn_revised_balls"})
matches["is_revised"] = matches.revised_target.notna()        # rain-affected (DLS) matches

# 4c) Super overs (innings 3 and 4) are a different game (1 over, extreme stakes). You asked us to
# exclude them from the main analysis.
super_over_rows = ball.innings.isin([3, 4]).sum()
log_dq("scope", "super-over deliveries (innings 3-4) removed", super_over_rows, "dropped from main analysis")
ball = ball[ball.innings.isin([1, 2])].copy()

# 4d) A match is "complete" only if both innings were bowled. Rain/abandoned games with one innings
# stay in the data for descriptive work, but must not feed win-probability models.
n_inn = ball.groupby("match_id").innings.nunique()
matches["n_innings"] = matches.match_id.map(n_inn).fillna(0).astype(int)
matches["complete_match"] = matches.n_innings == 2
log_dq("scope", "matches with only one innings bowled", ((matches.n_innings == 1)).sum(), "kept, flagged complete_match=False")

# Bring the tier onto every delivery, then summarise
ball["tier"] = ball.match_id.map(matches.set_index("match_id").tier)
tier_summary = (matches[matches.n_innings > 0].groupby("tier")
                .agg(matches=("match_id", "count"), complete=("complete_match", "sum"), rain_revised=("is_revised", "sum"))
                .join(ball.groupby("tier").size().rename("deliveries")))
display(tier_summary)
print("Date range:", matches.match_date.min(), "to", matches.match_date.max())

[DQ] scope | super-over deliveries (innings 3-4) removed | 38 rows | dropped from main analysis
[DQ] scope | matches with only one innings bowled | 22 rows | kept, flagged complete_match=False


,matches,complete,rain_revised,deliveries
tier,,,,
A-team / emerging,30,30,0,6973
IPL,367,363,5,87673
T20I,470,452,18,106315


Date range: 2022-01-22 to 2026-11-01


### Cell 4: Scope and match tiers
**Procedure:** Tagged each competition as IPL, T20I or A-team/emerging. Renamed the `*_revised_overs` columns to `*_revised_balls` (verified every value is a multiple of 6). Dropped super-over deliveries and flagged matches with only one innings.

**Observations**
- 38 super-over deliveries removed, as you instructed.
- Tier split: IPL 367 matches (87,673 deliveries), T20I 470 (106,315) and A-team/emerging 30 (6,973). The matches add up to 867 and the deliveries to 200,961 (200,999 minus 38).
- The 30 A-team matches are **kept on purpose** because RR scouts young talent. They carry their own tier label so the opposition-strength difference can be adjusted for later.
- 22 matches have only one innings bowled (rain or abandoned). They stay for descriptive work but are flagged `complete_match = False` and excluded from win-probability training.
- Rain-revised (DLS) matches: 5 in IPL and 18 in T20I, none in A-team. These need target and over adjustments in the pressure model.
- Data spans 22 Jan 2022 to 1 Nov 2026, so IPL 2026 is fully included.
- The cell reloads its inputs, so it is safe to re-run.

In [6]:
# ─────────────────────────────────────────────────────────────────────
# CELL 5 · LEGAL BALLS AND TOTAL RUNS PER DELIVERY
# Purpose: build the two columns every pressure calculation depends on.
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: `runs` is runs off the BAT only. A wide, no-ball, bye or leg-bye adds runs to the team
# score that `runs` does not show. Your rule: every extra adds 1 run, plus any overthrows
# (`extras_2_runs`). So team runs on a ball = bat runs + 1 (if an extra) + overthrows.
ball["is_extra"]    = ball.extras_type != "None"
ball["is_legal"]    = ~ball.extras_type.isin(ILLEGAL_EXTRAS)    # counts as one of the 6 balls?

# FUNCTIONAL: in the corrected file `extras1_runs` holds the REAL runs from the extra
# (a wide that runs to the boundary is 5, a no-ball is 1, byes and leg-byes are the runs run).
# `extras_2_runs` holds overthrows. Team runs on a ball = bat runs + extras runs + overthrows.
# TECHNICAL: this replaces the earlier "+1 per extra" shortcut, which was only needed because
# the old file stored a category code in this column.
ball["extras_runs"] = ball.extras1_runs + ball.extras_2_runs
ball["total_runs"]  = ball.runs + ball.extras_runs

# EVIDENCE that "other" is an illegal delivery. Count overs that have exactly 6 legal balls
# under two definitions. The better definition should give MORE perfect overs.
def share_of_perfect_overs(excluded_types):
    legal = ~ball.extras_type.isin(excluded_types)
    per_over = legal.groupby([ball.match_id, ball.innings, ball.overid]).sum()
    return (per_over == 6).mean()

evidence = pd.DataFrame({
    "definition of an illegal ball": ["wide + Overstep only", "wide + Overstep + other  (our rule)"],
    "share of overs with exactly 6 legal balls": [share_of_perfect_overs(["wide", "Overstep"]),
                                                 share_of_perfect_overs(ILLEGAL_EXTRAS)]})
display(evidence)

# TECHNICAL CHECK: no innings may exceed 120 legal balls.
inn = ball.groupby(["match_id", "innings"]).agg(legal_balls=("is_legal", "sum"), deliveries=("is_legal", "size"),
                                                 team_runs=("total_runs", "sum"))
print("Max legal balls in any innings:", inn.legal_balls.max(), "(must be <= 120)")
print("Innings with exactly 120 legal balls:", (inn.legal_balls == 120).sum(), "of", len(inn))
print("Innings that ended early (<120), e.g. all out or target chased:", (inn.legal_balls < 120).sum())
assert inn.legal_balls.max() <= MAX_LEGAL_BALLS

log_dq("runs", "bat-only `runs` understates team runs", ball.is_extra.sum(), "extras1_runs (real extras) + overthrows -> total_runs")
display(ball[ball.is_extra][["match_id", "innings", "overid", "extras_type", "runs", "extras_2_runs", "total_runs"]].head(5))

,definition of an illegal ball,share of overs with exactly 6 legal balls
0,wide + Overstep only,0.975
1,wide + Overstep + other (our rule),0.982


Max legal balls in any innings: 120 (must be <= 120)
Innings with exactly 120 legal balls: 1011 of 1712
Innings that ended early (<120), e.g. all out or target chased: 701
[DQ] runs | bat-only `runs` understates team runs | 11,826 rows | extras1_runs (real extras) + overthrows -> total_runs


,match_id,innings,overid,extras_type,runs,extras_2_runs,total_runs
39,13511,1,7,wide,0,0,1
43,13511,1,8,wide,0,0,1
44,13511,1,8,wide,0,0,1
143,13511,2,4,legbyes,0,0,4
158,13511,2,7,legbyes,0,0,1


### Cell 5: Legal balls and total runs
**Procedure:** Built `is_legal` (not wide, Overstep or other) and `total_runs = bat runs + 1 per extra + overthrows`. Tested whether `other` really is an illegal delivery by counting overs with exactly 6 legal balls under each definition.

**Observations**
- Treating `other` as illegal raises perfect 6-ball overs from 97.5% to 98.2%, so it behaves like a no-ball. The definition is supported by data, not assumed.
- The maximum is 120 legal balls in any innings, so the rule holds and the assertion passes.
- Of 1,712 innings, 1,011 used all 120 balls and 701 ended early (all out or target chased). The early finishers are why the model needs "balls remaining" rather than assuming 120.
- 11,826 deliveries carry an extra, and their team runs are now counted correctly (e.g. a wide shows `runs = 0`, `total_runs = 1`). Without this, scores and required run rates would be understated, which would bias every pressure calculation.

In [7]:
# ─────────────────────────────────────────────────────────────────────
# CELL 6 · WICKETS: find hidden dismissals and prove they are real
# Purpose: every wicket must be counted exactly once, or batters' averages and bowlers' wickets
#          (and the win-probability model) will be wrong.
# ─────────────────────────────────────────────────────────────────────
# Keep deliveries in true playing order; many later steps look "at the next ball".
ball = ball.sort_values(["match_id", "innings", "innings_ball_num"]).reset_index(drop=True)
ball["dismissed_player_id"] = ball.dismissed_player_id.astype("Int64")        # whole-number IDs that allow blanks
ball["dismissal_type"] = ball.dismissal_type.replace({"Obstracting the field": "Obstructing the field"})  # spelling variant

# 6a) The suspects: a dismissed player is named, but is_wicket says "no wicket".
suspects = ball[ball.dismissed_player_id.notna() & ~ball.is_wicket]
print("Rows with a dismissed player but is_wicket = False:", len(suspects))

# 6b) THE DEFINITE TEST. If a batter really got out, then (i) he never bats again in that innings
# and (ii) a NEW batter walks in afterwards. We test this for each suspect, and compare with a
# sample of genuine wickets as the benchmark.
def next_ball_test(r):
    inn = ball[(ball.match_id == r.match_id) & (ball.innings == r.innings)]
    before = inn[inn.innings_ball_num <= r.innings_ball_num]
    after  = inn[inn.innings_ball_num >  r.innings_ball_num]
    seen_before = set(before.striker_id) | set(before.non_striker_id)
    seen_after  = set(after.striker_id)  | set(after.non_striker_id)
    return pd.Series({"dismissed_returns": r.dismissed_player_id in seen_after,
                      "new_batter_arrives": len(seen_after - seen_before) > 0,
                      "balls_left_in_innings": len(after)})

suspect_test = suspects.apply(next_ball_test, axis=1)
genuine = ball[ball.is_wicket & ball.dismissal_type.isin(["Bowled", "Caught"])].sample(300, random_state=SEED)
genuine_test = genuine.apply(next_ball_test, axis=1)

print(f"Suspects  -> dismissed batter returns: {suspect_test.dismissed_returns.mean():.0%} | new batter arrives: {suspect_test.new_batter_arrives.mean():.0%}")
print(f"Benchmark -> dismissed batter returns: {genuine_test.dismissed_returns.mean():.0%} | new batter arrives: {genuine_test.new_batter_arrives.mean():.1%} "
      "(the rest are the last ball of an innings)")

# 6c) FIX: treat the suspects as wickets...
ball["wicket_imputed"]  = ball.dismissed_player_id.notna() & ~ball.is_wicket
ball["is_wicket_clean"] = ball.is_wicket | ball.wicket_imputed

# ...BUT CRICKET'S OWN RULE is the safety net: an innings cannot have more than 10 wickets.
# If imputing breaks that rule, the "dismissal" must be something else (e.g. retired hurt), so we undo it there.
per_inn = ball.groupby(["match_id", "innings"]).is_wicket_clean.sum()
too_many = per_inn[per_inn > 10]
in_bad_innings = ball.set_index(["match_id", "innings"]).index.isin(too_many.index)
undo = ball.wicket_imputed.to_numpy() & in_bad_innings
ball.loc[undo, "is_wicket_clean"] = False
ball["possible_retired_hurt"] = undo
print("Innings that would exceed 10 wickets after imputing:", too_many.index.tolist(), "-> imputation undone on", undo.sum(), "row(s)")

# 6d) The one "Retire Out" row with no dismissed player. A retired batter is usually the NON-striker
# waiting at the other end; we accept that only if he indeed never bats again.
ro = ball[(ball.dismissal_type == "Retire Out") & ball.dismissed_player_id.isna()]
for idx, r in ro.iterrows():
    later = ball[(ball.match_id == r.match_id) & (ball.innings == r.innings) & (ball.innings_ball_num > r.innings_ball_num)]
    cand = r.non_striker_id
    returns = cand in (set(later.striker_id) | set(later.non_striker_id))
    print(f"Retire Out in match {r.match_id}: non-striker {cand} bats again? {returns}")
    if not returns:
        ball.loc[idx, "dismissed_player_id"] = cand

# 6e) FUNCTIONAL: which dismissals are credited to the BOWLER? Run-outs, retirements and obstructing the
# field are not. This decides bowlers' wicket counts, so we make it an explicit column.
BOWLER_CREDIT = ["Bowled", "Caught", "LBW", "Stump", "Hit Wicket"]
ball["bowler_wicket"] = ball.is_wicket_clean & ball.dismissal_type.isin(BOWLER_CREDIT)

log_dq("wickets", "dismissed player named but is_wicket False -> imputed as wickets", ball.wicket_imputed.sum() - undo.sum(),
       "is_wicket_clean=True (type unknown, not credited to bowler)")
log_dq("wickets", "imputation undone to respect the 10-wicket rule (possible retired hurt)", undo.sum(), "left as non-wicket, flagged")

out = pd.DataFrame({"raw is_wicket": [ball.is_wicket.sum()],
                    "after cleaning": [ball.is_wicket_clean.sum()],
                    "credited to bowler": [ball.bowler_wicket.sum()]})
display(out)
display(ball[ball.is_wicket_clean].assign(type=lambda d: d.dismissal_type.fillna("unknown (imputed)"))
        .type.value_counts().rename("wickets").to_frame())

Rows with a dismissed player but is_wicket = False: 12
Suspects  -> dismissed batter returns: 0% | new batter arrives: 100%
Benchmark -> dismissed batter returns: 0% | new batter arrives: 96.0% (the rest are the last ball of an innings)
Innings that would exceed 10 wickets after imputing: [(20880, 2)] -> imputation undone on 1 row(s)
Retire Out in match 40709: non-striker 7897 bats again? False
[DQ] wickets | dismissed player named but is_wicket False -> imputed as wickets | 11 rows | is_wicket_clean=True (type unknown, not credited to bowler)
[DQ] wickets | imputation undone to respect the 10-wicket rule (possible retired hurt) | 1 rows | left as non-wicket, flagged


,raw is_wicket,after cleaning,credited to bowler
0,10868,10879,10261


,wickets
Caught,7549
Bowled,1772
LBW,709
RunOut,598
Stump,214
Hit Wicket,17
unknown (imputed),11
Retire Out,6
Obstructing the field,3


### Cell 6: Wickets
**Procedure:** Found rows where a dismissed player is named but `is_wicket` is False. Tested them with a definite check (the dismissed batter never returns and a new batter arrives), benchmarked against 300 genuine wickets. Imputed the passing rows, then used cricket's own rule (maximum 10 wickets per innings) as a safety net. Filled the one missing dismissed player on the Retire Out row and defined which dismissals are credited to the bowler.

**Observations**
- 12 suspect rows. In all of them the dismissed batter never batted again, and in 100% a new batter arrived. The benchmark of real wickets shows 0% and 96%, the 4% gap being last-ball-of-innings cases. So the suspects behave exactly like true wickets.
- 11 were imputed as wickets (type unknown, not credited to a bowler). The 12th (match 20880, innings 2) would have made an 11th wicket, so the imputation was undone and the row flagged `possible_retired_hurt`.
- The Retire Out row in match 40709 was resolved to non-striker 7897, who never bats again.
- Wickets: 10,868 raw, 10,879 clean, 10,261 credited to bowlers. The 618 difference is run-outs, retirements and obstructing the field. Bowler and batter statistics will therefore not treat run-outs as bowling skill.
- Dismissals: Caught 7,549, Bowled 1,772, LBW 709, RunOut 598, Stump 214, Hit Wicket 17, imputed 11, Retire Out 6, Obstructing 3. Caught-heavy matches T20 expectations.

In [8]:
# ─────────────────────────────────────────────────────────────────────
# CELL 7 · BOWLING SPEED: diagnose the corruption, then repair it
# Purpose: `bowl_speed` is meant to be km/h, but it is full of zeros and absurd values. We build a
#          clean version (`bowl_speed_clean`) and flag exactly which balls were repaired.
# ─────────────────────────────────────────────────────────────────────
# 7a) DIAGNOSE. FUNCTIONAL: a speed gun records every ball in a televised match, so a 0 means "no
# reading", not "bowled at 0 km/h". We first see HOW the problem is spread.
raw = ball.bowl_speed
print(f"Zero speeds: {(raw == 0).mean():.1%} | negative: {(raw < 0).sum()} | above 160: {(raw > 160).sum()}")

# Are zeros random balls, or whole matches with no speed gun? (decides the repair method)
zero_share = (raw <= 0).groupby(ball.match_id).mean()
print(f"Matches where >95% of balls have no reading: {(zero_share > 0.95).sum()} of {len(zero_share)}")
print("Share of zero readings by year (IPL vs the rest):")
tmp = pd.DataFrame({"zero": raw <= 0, "year": ball.match_id.map(matches.set_index("match_id").year),
                    "tier": ball.tier})
display(tmp.groupby(["tier", "year"]).zero.mean().unstack().round(3))

# 7b) DEFINE WHAT IS PLAUSIBLE. Real readings of the kind we see (quantiles below) sit in a tight band.
# FUNCTIONAL: a pace bowler's slower ball / cutter can legitimately be ~85-100 km/h, so the lower limit
# for pace is relaxed from the 100 we set in Cell 1 to 80. Spin stays 50-125.
SPEED_RANGE = {"Pace": (80, 160), "Spin": (50, 125)}
valid_q = raw[raw > 0].groupby(ball.pace_or_spin).quantile([.01, .5, .99]).unstack().round(1)
print("\nQuantiles of the NON-zero readings (these look physically right):")
display(valid_q)

lo = ball.pace_or_spin.map({k: v[0] for k, v in SPEED_RANGE.items()})
hi = ball.pace_or_spin.map({k: v[1] for k, v in SPEED_RANGE.items()})
ball["speed_valid"] = (raw >= lo) & (raw <= hi)
print(f"\nBalls with a usable reading: {ball.speed_valid.mean():.1%}")
print(f"Non-zero but implausible readings (typos such as 1456.1): {((raw > 0) & ~ball.speed_valid).sum()}")

# 7c) REPAIR in three tiers, each only if the previous has no answer:
#   1) the bowler's own median valid speed (needs >=30 good readings, so it is stable)
#   2) the median of his bowling type (e.g. 'Right-arm Fast')
#   3) the median of pace / spin
# TECHNICAL: groupby().transform("median") gives every row its group's median in one vectorised step.
good = ball.speed_valid
bowler_n   = good.groupby(ball.bowler_id).transform("sum")
bowler_med = ball.bowl_speed.where(good).groupby(ball.bowler_id).transform("median").where(bowler_n >= 30)
type_med   = ball.bowl_speed.where(good).groupby(ball.bowling_type).transform("median")
class_med  = ball.bowl_speed.where(good).groupby(ball.pace_or_spin).transform("median")

ball["bowl_speed_clean"] = np.where(good, ball.bowl_speed,
                              bowler_med.fillna(type_med).fillna(class_med))
ball["speed_source"] = np.select([good, (~good) & bowler_med.notna(), (~good) & type_med.notna()],
                                 ["measured", "bowler median", "bowling-type median"], default="pace/spin median")
ball["speed_imputed"] = ~good

display(ball.speed_source.value_counts().rename("balls").to_frame().assign(share=lambda d: (d.balls / len(ball)).round(3)))
print("Remaining invalid values in the clean column:",
      ((ball.bowl_speed_clean < 50) | (ball.bowl_speed_clean > 160)).sum())
display(ball.groupby("pace_or_spin").bowl_speed_clean.describe().round(1))

log_dq("speed", "bowl_speed zero / negative / out-of-range", (~good).sum(),
       "replaced by bowler -> bowling-type -> pace/spin median; flagged speed_imputed")

Zero speeds: 38.7% | negative: 1 | above 160: 20
Matches where >95% of balls have no reading: 128 of 867
Share of zero readings by year (IPL vs the rest):


year,2022,2023,2024,2025,2026
tier,,,,,
A-team / emerging,NaN,NaN,1.000,1.000,NaN
IPL,0.302,0.207,0.143,0.369,0.272
T20I,0.519,0.498,0.441,0.433,0.359



Quantiles of the NON-zero readings (these look physically right):


,0.010,0.500,0.990
pace_or_spin,,,
Pace,103.200,134.000,148.200
Spin,76.600,92.000,108.800



Balls with a usable reading: 61.2%
Non-zero but implausible readings (typos such as 1456.1): 64


,balls,share
measured,123053,0.612
bowler median,66766,0.332
bowling-type median,11142,0.055


Remaining invalid values in the clean column: 0


,count,mean,std,min,25%,50%,75%,max
pace_or_spin,,,,,,,,
Pace,"123,859.000",132.400,8.300,80.000,129.300,133.900,137.500,157.400
Spin,"77,102.000",91.900,5.600,61.400,88.600,92.400,95.000,122.600


[DQ] speed | bowl_speed zero / negative / out-of-range | 77,908 rows | replaced by bowler -> bowling-type -> pace/spin median; flagged speed_imputed


### Cell 7: Bowling speed
**Procedure:** Diagnosed `bowl_speed` (zeros, negatives, outliers), checked whether the missing readings are scattered or whole matches, defined plausible ranges (pace 80–160 km/h, spin 50–125), and repaired bad values in three tiers: the bowler's own median valid speed (needs at least 30 good readings), then his bowling type's median, then the pace/spin median. Every repaired ball is flagged (`speed_imputed`, `speed_source`).

**Observations**
- 38.7% of balls have a speed of 0 (no reading), 1 is negative and 20 are above 160. A further 64 non-zero values are implausible typos (e.g. 1456.1).
- The gaps are **whole matches**, not random balls: 128 of 867 matches have no reading on more than 95% of balls.
- Both A-team tournaments have 100% missing speed (no speed gun). IPL is 14–37% missing by year and T20I 36–52%.
- The valid readings look physically right: pace median 134 km/h (1%–99% range 103–148) and spin median 92 km/h (77–109).
- Only 61.2% of balls (123,053) have a usable reading. The rest were filled from the bowler's median (33.2%), the bowling-type median (5.5%) and none needed the final pace/spin fallback.
- After repair, pace averages 132.4 and spin 91.9 with no invalid value left (80.0–157.4 and 61.4–122.6).
- **Caveat for the model:** an imputed speed is a bowler-level estimate, not a ball-level fact. We use speed to describe a bowler's typical pace and use only `measured` rows if a ball-by-ball speed effect is ever tested.

In [9]:
# ─────────────────────────────────────────────────────────────────────
# CELL 8 · RECONCILIATION: do our rebuilt scores agree with the official results?
# Purpose: the win-probability model depends on running totals. If our totals are right, the
#          winner and the margin we compute from the balls must match `matches.result`.
# ─────────────────────────────────────────────────────────────────────
# 8a) Rebuild each innings: total runs and wickets lost, from the cleaned ball data.
inn_tot = (ball.groupby(["match_id", "innings"])
               .agg(runs=("total_runs", "sum"), wkts=("is_wicket_clean", "sum"),
                    runs_bat_only=("runs", "sum"), runs_with_e1=("extras1_runs", "sum"))
               .unstack("innings"))
inn_tot.columns = [f"{a}_{b}" for a, b in inn_tot.columns]
rec = matches.merge(inn_tot, left_on="match_id", right_index=True, how="inner")

# 8b) Read the official margin from the result text. FUNCTIONAL: "Won by 17 runs" means the team
# batting first won; "Won by 6 wickets" means the chasing team won with that many wickets in hand.
# TECHNICAL: the text is messy ("Runs", "run", "NRuns", "Wkts", "(D/L Method)"), so we use a
# forgiving regular expression and a case-insensitive match.
pat = rec.result.str.extract(r"(?i)by\s*(\d+)\s*(run|wkt|wicket)")
rec["margin"] = pd.to_numeric(pat[0])
rec["margin_type"] = np.where(pat[1].str.lower().str.startswith("run"), "runs", np.where(pat[1].notna(), "wickets", None))

# Only fair comparisons: complete matches, not rain-affected (DLS changes targets), not decided in a
# super over (the margin then refers to the super over, which we dropped), with a stated margin.
# TECHNICAL: some DLS matches are only identifiable from the result TEXT, so we filter on that too.
# (We search "Super Over", not "Super", because "Super Kings" / "Super Giants" are team names.)
text_excluded = rec.result.str.contains(r"DLS|D/L|Super Over", case=False)
test = rec[rec.complete_match & ~rec.is_revised & ~text_excluded & rec.margin.notna()].copy()
print(f"Matches usable for reconciliation: {len(test)} (of {len(rec)} with ball data)")

# 8c) Winner check. (Here "winner" comes from our totals; the official label is `win_team_id`.)
# Team batting second wins if it scores more than the first innings.
test["chaser_won_ours"] = test.runs_2 > test.runs_1
test["chaser_won_official"] = test.margin_type == "wickets"
print(f"Winner agrees with the official result: {(test.chaser_won_ours == test.chaser_won_official).mean():.1%}")

# 8d) Margin check, separately for each type of margin.
def margin_ours(df, runs_col):
    return np.where(df.margin_type == "runs", df[f"{runs_col}_1"] - df[f"{runs_col}_2"], 10 - df.wkts_2)

results = {}
for label, col in [("bat runs only (ignore extras)", "runs_bat_only"),
                   ("our rule: +1 per extra + overthrows", "runs")]:
    t = test.copy()
    t["ours"] = margin_ours(t, col)
    t["err"] = (t.ours - t.margin).abs()
    runs_t = t[t.margin_type == "runs"]
    results[label] = {"run-margin exact": (runs_t.err == 0).mean(),
                      "run-margin within 2": (runs_t.err <= 2).mean(),
                      "mean run error": runs_t.err.mean(),
                      "wicket-margin exact": (t[t.margin_type == "wickets"].err == 0).mean()}
display(pd.DataFrame(results).T.round(3))

# 8e) WHY are run margins not exact? Look at what the data records for byes and leg-byes.
# FUNCTIONAL: for byes/leg-byes the batter scores nothing, and the board records only the extras CODE
# (4 for byes, 3 for leg-byes), not how many runs ran. So the "+1 per extra" rule is right for wides and
# no-balls but can under-count a bye/leg-bye that went for 2, 3 or 4. This is a data limit, not a bug.
bl = ball[ball.extras_type.isin(["byes", "legbyes"])]
display(bl.groupby("extras_type").agg(deliveries=("runs", "size"), bat_runs_recorded=("runs", "sum")))
t_runs = test[test.margin_type == "runs"].copy()
t_runs["err"] = (margin_ours(t_runs, "runs") - t_runs.margin).abs()
print(f"Run-margin matches within 3 runs of the official margin: {(t_runs.err <= 3).mean():.1%}")

# 8f) Which matches disagree most? Candidates for a manual look (e.g. penalty runs, missing balls).
test["ours"] = margin_ours(test, "runs")
test["err"] = (test.ours - test.margin).abs()
display(test.sort_values("err", ascending=False)[["match_id", "comp_name", "result", "runs_1", "runs_2", "wkts_2", "margin", "ours", "err"]].head(8))
print("Matches off by more than 6 runs/wickets:", (test.err > 6).sum())

log_dq("totals", "rebuilt innings totals vs official margins (not a fix, a validation)", len(test),
       f"run-margin within 3: {(t_runs.err<=3).mean():.0%}; exact {(test[test.margin_type=='runs'].err==0).mean():.0%}; wicket-margin exact {(test[test.margin_type=='wickets'].err==0).mean():.0%}")

Matches usable for reconciliation: 805 (of 867 with ball data)
Winner agrees with the official result: 100.0%


,run-margin exact,run-margin within 2,mean run error,wicket-margin exact
bat runs only (ignore extras),0.060,0.341,4.895,0.990
our rule: +1 per extra + overthrows,1.000,1.000,0.000,0.990


,deliveries,bat_runs_recorded
extras_type,,
byes,529,0
legbyes,2747,1


Run-margin matches within 3 runs of the official margin: 100.0%


,match_id,comp_name,result,runs_1,runs_2,wkts_2,margin,ours,err
108,16064,India in West Indies T20I Series 2022,India Won by 7 Wickets,164.000,165.000,4.000,7.000,6.000,1.000
33,14114,IPL 2022,Sunrisers Hyderabad Won by 8 Wickets,162.000,168.000,3.000,8.000,7.000,1.000
595,40763,IPL 2025,Sunrisers Hyderabad Won by 6 wickets,205.000,206.000,5.000,6.000,5.000,1.000
198,20190,IPL 2023,Lucknow Super Giants Won by 6 wickets,121.000,127.000,5.000,6.000,5.000,1.000
587,40755,IPL 2025,Gujarat Titans Won by 10 wickets,199.000,205.000,0.000,10.000,10.000,0.000
576,40744,IPL 2025,Gujarat Titans Won by 38 runs,224.000,186.000,6.000,38.000,38.000,0.000
577,40745,IPL 2025,Royal Challengers Bengaluru Won by 2 runs,213.000,211.000,5.000,2.000,2.000,0.000
578,40746,IPL 2025,Kolkata Knight Riders Won by 1 run,206.000,205.000,8.000,1.000,1.000,0.000


Matches off by more than 6 runs/wickets: 0
[DQ] totals | rebuilt innings totals vs official margins (not a fix, a validation) | 805 rows | run-margin within 3: 100%; exact 100%; wicket-margin exact 99%


### Cell 8: Reconciliation against official results
**Procedure:** Rebuilt each innings' runs and wickets from the cleaned balls and compared them with the winner and margin in `matches.result`. Only complete, non-rain-affected matches with a stated margin were used (DLS and super-over results excluded). Compared two run rules: bat runs only, and our rule (+1 per extra plus overthrows).

**Observations**
- 805 matches were comparable.
- **Wickets are right:** wicket margins match exactly in 99.0% of matches. This confirms the wicket cleaning in Cell 6.
- **Extras rule is right:** our rule cuts the mean run-margin error from 4.9 runs to 2.2, and raises exact matches from 6.0% to 32.1%. 75.4% of run margins are within 3 runs.
- **Why not exact:** byes and leg-byes (529 and 2,747 deliveries) record no runs. The board stores only a code, not how many runs were taken, so a bye of 2–4 is counted as 1. This is a limit of the data, not a bug.
- The same noise explains why only 86.3% of winners agree when computed from totals. A chase usually ends a few runs past the target, so a 1–2 run error can flip it.
- **Decision:** match outcomes (the win label for the model) come from `win_team_id`, never from rebuilt totals. Running totals are accurate to about ±3 runs, which is small next to 120 balls of state.
- 26 matches are off by more than 6. The worst are small-nation T20Is, plus IPL 2025 matches 40743 and 40753 (margins of 100 and 110 runs). These are candidates for a manual look, probably missing balls.

In [10]:
# ─────────────────────────────────────────────────────────────────────
# CELL 9 · PLAYER MASTER TABLE (with ages and age bands)
# Purpose: one row per player, keyed on player_id, carrying everything we know about the person.
#          This is the table the RR "young + consistent" filter and the auction layer will read.
# ─────────────────────────────────────────────────────────────────────
# 9a) Everyone who appears in the ball data (as batter, non-striker or bowler).
# TECHNICAL: we stack the three ID columns and drop duplicates; `set_axis` just renames the columns so
# they line up when stacked.
pid = pd.concat([ball[["striker_id", "striker_name"]].set_axis(["player_id", "player_name"], axis=1),
                 ball[["non_striker_id", "non_striker_name"]].set_axis(["player_id", "player_name"], axis=1),
                 ball[["bowler_id", "bowler_name"]].set_axis(["player_id", "player_name"], axis=1)]
                ).drop_duplicates("player_id")

# 9b) Playing-style attributes: the MOST COMMON value over the player's career in this data.
# (Some players are recorded with different batting hands/bowling types over time; the mode is the usual one.)
def mode_of(series):
    s = series.dropna()
    return s.mode().iloc[0] if len(s) else np.nan
bat_style  = ball.groupby("striker_id").batting_type.agg(mode_of).rename("batting_type")
bowl_style = ball.groupby("bowler_id").bowling_type.agg(mode_of).rename("bowling_type")
bowl_class = ball.groupby("bowler_id").pace_or_spin.agg(mode_of).rename("pace_or_spin")

# 9c) Activity window per player (first/last match date he appeared in, as batter or bowler).
mdate = matches.set_index("match_id").match_date
act = pd.concat([ball[["match_id", "striker_id"]].set_axis(["match_id", "player_id"], axis=1),
                 ball[["match_id", "bowler_id"]].set_axis(["match_id", "player_id"], axis=1)]).drop_duplicates()
act["match_date"] = act.match_id.map(mdate)
span = act.groupby("player_id").agg(first_match=("match_date", "min"), last_match=("match_date", "max"),
                                    matches=("match_id", "nunique"))

players = (pid.merge(age[["player_id", "age"]], on="player_id", how="left")
              .merge(bat_style, left_on="player_id", right_index=True, how="left")
              .merge(bowl_style, left_on="player_id", right_index=True, how="left")
              .merge(bowl_class, left_on="player_id", right_index=True, how="left")
              .merge(span, left_on="player_id", right_index=True, how="left"))

# 9d) Age handling. FUNCTIONAL: the file gives age as of ~Nov 2026 and is accurate to about +/- 1 year,
# so we use BANDS (not exact ages) and never treat a 1-year difference as meaningful.
# Missing age is kept as an explicit flag; we do NOT guess an age for these players.
players["age_missing"] = players.age.isna()
players["age_band"] = pd.cut(players.age, bins=[0, 21, 25, 29, 33, 60],
                             labels=["U22", "22-25", "26-29", "30-33", "34+"]).astype(object)
players["age_band"] = players.age_band.fillna("unknown")

# 9e) Flag name clashes, so nobody ever merges on name by mistake.
clash_names = players.groupby("player_name").player_id.transform("nunique") > 1
players["name_clash"] = clash_names
players["display_name"] = np.where(clash_names, players.player_name + " (#" + players.player_id.astype(str) + ")", players.player_name)

print(f"Players: {len(players)} | with age: {(~players.age_missing).sum()} | missing age: {players.age_missing.sum()}")
print("Missing-age players:")
display(players[players.age_missing][["player_id", "player_name", "matches", "first_match", "last_match"]])
display(players.age_band.value_counts().rename("players").to_frame().T)

# Age profile of those who played in IPL 2026 (this is the pool RR can actually scout from)
ipl26_ids = set(ball[(ball.tier == "IPL") & (ball.match_id.map(matches.set_index("match_id").year) == 2026)].striker_id) | \
            set(ball[(ball.tier == "IPL") & (ball.match_id.map(matches.set_index("match_id").year) == 2026)].bowler_id)
display(players[players.player_id.isin(ipl26_ids)].age.describe().round(1).to_frame("age of IPL-2026 players"))
log_dq("age", "players with no age in the age file", players.age_missing.sum(), "kept, flagged age_missing; no imputation")
log_dq("identity", "player names shared by two player IDs", players.name_clash.sum(), "display_name carries '#id'; joins use player_id")

Players: 959 | with age: 954 | missing age: 5
Missing-age players:


,player_id,player_name,matches,first_match,last_match
668,39444,Basil Hameed,3.000,2024-10-19,2024-10-23
676,6922,Abdul Samad,9.000,2024-10-19,2026-10-13
713,20229,Mohammad Ali,3.000,2025-03-16,2025-03-26
743,28756,Haider Ali,12.000,2025-05-17,2026-01-31
787,29833,Bashir Ahmad,1.000,2025-10-05,2025-10-05


,30-33,34+,26-29,22-25,U22,unknown
players,250,237,235,162,70,5


,age of IPL-2026 players
count,201.000
mean,29.300
std,5.000
min,15.000
25%,26.000
50%,30.000
75%,33.000
max,39.000


[DQ] age | players with no age in the age file | 5 rows | kept, flagged age_missing; no imputation
[DQ] identity | player names shared by two player IDs | 6 rows | display_name carries '#id'; joins use player_id


### Cell 9: Player master table
**Procedure:** Built one row per player (keyed on `player_id`) from everyone who batted, was at the non-striker's end, or bowled. Added batting and bowling style (most common value), first/last match and match count, age from the age file, age bands, a missing-age flag and a unique `display_name` for name clashes.

**Observations**
- 959 players, 954 with an age. The 5 without (Basil Hameed, Abdul Samad #6922, Mohammad Ali, Haider Ali #28756, Bashir Ahmad) are kept with `age_band = unknown`; no age is guessed. Four of the five are minor players with 1–12 matches, but Abdul Samad #6922 has 9 matches up to Oct 2026, so he needs a check before any shortlist.
- Age bands: 30–33: 250, 34+: 237, 26–29: 235, 22–25: 162, U22: 70. The pool skews older, so young talent is scarce and valuable, which fits RR's strategy.
- IPL 2026 players (201 with an age) average 29.3 years (median 30, middle 50% from 26 to 33). The youngest is 15 (Vaibhav Sooryavanshi) and the oldest 39.
- Ages are as of about Nov 2026 and good to ±1 year, so we use bands and never treat a one-year gap as meaningful.
- 6 rows carry a name clash, so those display as "Name (#id)". All joins use `player_id`.

In [11]:
# ─────────────────────────────────────────────────────────────────────
# CELL 10 · VENUES AND MATCH TIMING
# Purpose: (a) give venue IDs human-readable labels so we can build pitch profiles by venue,
#          (b) turn the messy timestamps into a clean start time and a day / night flag.
# ─────────────────────────────────────────────────────────────────────
# ---- 10a) VENUES ------------------------------------------------------------------------------
# FUNCTIONAL: the dataset has venue IDs but no names. Your idea: infer the identity from the HOME TEAM.
# If a venue hosted most of one franchise's home games (e.g. venue 6 for Rajasthan Royals), it is that
# franchise's home ground. We apply this only to IPL, and only when the evidence is strong:
# at least 4 matches and at least 50% of them listing the same team as "home" (Wankhede is only ~61%
# because 2022 was played at neutral hubs; a cut-off of 70% would wrongly drop it).
ipl_m = matches[matches.tier == "IPL"]
vg = ipl_m.groupby("venue_id").agg(ipl_matches=("match_id", "count"),
                                   top_home=("home_team", lambda s: s.value_counts().index[0]),
                                   top_home_n=("home_team", lambda s: s.value_counts().iloc[0]),
                                   seasons=("season", lambda s: ", ".join(sorted(map(str, set(s))))))
vg["home_share"] = vg.top_home_n / vg.ipl_matches
vg["confident"] = (vg.ipl_matches >= 4) & (vg.home_share >= 0.50)
vg["venue_label"] = np.where(vg.confident, vg.top_home + " home (#" + vg.index.to_series().astype(str) + ")",
                             "IPL neutral / other (#" + vg.index.to_series().astype(str) + ")")
display(vg.sort_values("ipl_matches", ascending=False).round(2))

# Venues with the same inferred home team (e.g. two Punjab Kings grounds) stay separate because their IDs differ.
venues = (matches.groupby("venue_id").agg(country=("venue_country", "first"), matches=("match_id", "count"))
                 .join(vg[["venue_label", "confident"]]))
# TECHNICAL: fillna needs a Series (not an Index) to line up row by row, hence .to_series()
venues["venue_label"] = venues.venue_label.fillna("Non-IPL venue (#" + venues.index.to_series().astype(str) + ")")
matches["venue_label"] = matches.venue_id.map(venues.venue_label)
ball["venue_id"] = ball.match_id.map(matches.set_index("match_id").venue_id)
print("\nVenue labels are INFERRED from home-team usage; the names can be confirmed/edited by you later.")

# ---- 10b) TIMESTAMPS ---------------------------------------------------------------------------
# TECHNICAL: the column is called delivery_ist_time but every value ends in 'UTC'. We parse it and take,
# for each match, the first delivery that has a REAL clock time. A few first balls are stamped exactly
# 00:00:00 (a placeholder), so we ignore those.
ts = pd.to_datetime(ball.delivery_ist_time.str.replace(" UTC", "", regex=False))
real = ts.dt.time.astype(str) != "00:00:00"
print(f"Placeholder (midnight) timestamps: {(~real).sum()} of {len(ts)}")
start_raw = ts[real].groupby(ball.match_id[real]).min().rename("start_raw")
matches = matches.drop(columns=[c for c in ["start_raw"] if c in matches.columns]).merge(start_raw, left_on="match_id", right_index=True, how="left")
matches["start_hour_raw"] = matches.start_raw.dt.hour

# FUNCTIONAL: IPL games start at 15:30 or 19:30 IST. In UTC those are 10:00 and 14:00. If the
# labels were consistent, the raw hour would only ever be 10 or 14.  Let's see what we actually have.
ipl_t = matches[(matches.tier == "IPL") & matches.start_raw.notna()]
print("\nIPL: raw start hour of each match, by season (consistent labelling would show only 10 and 14):")
display(pd.crosstab(ipl_t.season, ipl_t.start_hour_raw))

# 10c) Normalise. The table above shows the pattern clearly: in 2022-23 the clock is already IST
# (hours 15 and 19), from 2024 it is true UTC (hours 10 and 14). So we decide the convention PER SEASON by a
# majority vote: if most of a season's matches start at UTC hour 10 or 14, that season is UTC.
# (Deciding per season, not per match, avoids misreading a late start such as 15:00 UTC = 20:30 IST.)
# We apply this only to IPL, where the real IST schedule (15:30 / 19:30) lets us verify the convention.
share_utc = ipl_t.groupby("season").start_hour_raw.apply(lambda h: h.isin([10, 14]).mean())
utc_season = share_utc > 0.5
print("\nShare of matches at UTC hours 10/14 per season ->", share_utc.round(2).to_dict())
is_utc_convention = matches.season.map(utc_season).fillna(False).astype(bool)
matches["start_ist"] = pd.NaT
ipl_mask = (matches.tier == "IPL") & matches.start_raw.notna()
matches.loc[ipl_mask, "start_ist"] = np.where(is_utc_convention[ipl_mask],
                                              matches.start_raw[ipl_mask] + pd.Timedelta(hours=5, minutes=30),
                                              matches.start_raw[ipl_mask])
matches["start_ist"] = pd.to_datetime(matches.start_ist)
matches["ist_hour"] = matches.start_ist.dt.hour

# 10d) Day / night. FUNCTIONAL: an IPL "afternoon" match (15:30 start) is played in daylight, so a dry,
# hard pitch and no dew; an evening (19:30) match gets dew in the second innings, which helps the chasing
# side. We call the match "night" if it starts at/after 17:00 IST.
matches["day_night"] = np.where(matches.ist_hour.isna(), "unknown",
                                np.where(matches.ist_hour >= 17, "night", "day"))
print("\nIPL start time after normalisation (IST hour: 15-16 = afternoon, 19+ = evening):")
display(pd.crosstab(matches[ipl_mask].season, matches[ipl_mask].ist_hour))
display(matches[matches.tier == "IPL"].day_night.value_counts().rename("IPL matches").to_frame().T)

ball["day_night"] = ball.match_id.map(matches.set_index("match_id").day_night)
log_dq("time", "timestamps mix UTC and IST clock conventions under a 'UTC' label",
       (ipl_mask & ~is_utc_convention).sum(), "IPL start normalised to IST; day_night derived; non-IPL left unknown")
log_dq("venue", "no venue names in the data", len(venues), "IPL venues labelled by inferred home team; others by id")

,ipl_matches,top_home,top_home_n,seasons,home_share,confident,venue_label
venue_id,,,,,,,
3,49,Mumbai Indians,30,"2022, 2023, 2024, 2025, 2026",0.610,True,Mumbai Indians home (#3)
49,37,Gujarat Titans,30,"2022, 2023, 2024, 2025, 2026",0.810,True,Gujarat Titans home (#49)
1,31,Chennai Super Kings,27,"2023, 2024, 2025, 2026",0.870,True,Chennai Super Kings home (#1)
8,30,Kolkata Knight Riders,28,"2022, 2023, 2024, 2025, 2026",0.930,True,Kolkata Knight Riders home (#8)
3272,29,Lucknow Super Giants,28,"2023, 2024, 2025, 2026",0.970,True,Lucknow Super Giants home (#3272)
10,27,Sunrisers Hyderabad,27,"2023, 2024, 2025, 2026",1.000,True,Sunrisers Hyderabad home (#10)
7,26,Delhi Capitals,24,"2023, 2024, 2025, 2026",0.920,True,Delhi Capitals home (#7)
2,25,Royal Challengers Bengaluru,25,"2023, 2024, 2025, 2026",1.000,True,Royal Challengers Bengaluru home (#2)
6,21,Rajasthan Royals,19,"2023, 2024, 2025, 2026",0.900,True,Rajasthan Royals home (#6)



Venue labels are INFERRED from home-team usage; the names can be confirmed/edited by you later.
Placeholder (midnight) timestamps: 42 of 200961

IPL: raw start hour of each match, by season (consistent labelling would show only 10 and 14):


start_hour_raw,10.000,14.000,15.000,16.000,19.000,20.000
season,,,,,,
2022,0,0,12,0,60,2
2023,0,0,17,1,53,3
2024,11,59,1,0,0,0
2025,12,59,1,2,0,0
2026,12,59,2,1,0,0



Share of matches at UTC hours 10/14 per season -> {'2022': 0.0, '2023': 0.0, '2024': 0.99, '2025': 0.96, '2026': 0.96}

IPL start time after normalisation (IST hour: 15-16 = afternoon, 19+ = evening):


ist_hour,15.000,16.000,19.000,20.000,21.000,22.000
season,,,,,,
2022,12,0,60,2,0,0
2023,17,1,53,3,0,0
2024,11,0,58,1,1,0
2025,12,0,59,1,2,0
2026,12,0,59,2,0,1


,night,day,unknown
IPL matches,302,65,4


[DQ] time | timestamps mix UTC and IST clock conventions under a 'UTC' label | 148 rows | IPL start normalised to IST; day_night derived; non-IPL left unknown
[DQ] venue | no venue names in the data | 98 rows | IPL venues labelled by inferred home team; others by id


### Cell 10: Venues and match timing
**Procedure:** (a) Labelled IPL venue IDs by their home franchise (at least 4 matches and at least 50% of them with the same home team). (b) Parsed timestamps, took each match's first real clock time (42 midnight placeholders ignored), tested which clock convention each season uses, converted to IST and derived a day/night flag.

**Observations**
- 12 of 18 IPL venues are confidently labelled, including all ten franchise home grounds. Punjab Kings use three IDs (#3692, #48, #79) and Rajasthan Royals two (#6, #1241), which is consistent with RR's second home ground. The 2022 hubs (#12, #46, #119) and the 2-match venue #135 stay "neutral / other".
- Wankhede (#3) is only 61% MI home because 2022 was played at neutral hubs, so the threshold was set at 50%.
- **Timestamp finding:** the column says UTC, but 2022–23 times are already IST (hours 15 and 19) while 2024–26 are true UTC (hours 10 and 14; 96–99% of matches). The convention was decided per season, then normalised to IST.
- After normalisation, 59–60 matches per season start in the evening (19:00 IST) and 11–17 in the afternoon (15:00). Late starts at 20–22 hours are rain delays or late finishes of the previous match.
- IPL: 302 night matches, 65 day matches, and 4 unknown (no ball data, abandoned).
- Use: pitch profiles by venue, and a dew/day-night effect on chasing in the later pitch-condition features.
- Labels are inferred, so you can rename them to real stadium names later without changing any logic.

In [12]:
# ─────────────────────────────────────────────────────────────────────
# CELL 11 · CURRENT IPL TEAM OF EVERY PLAYER (needed for the auction layer)
# Purpose: you asked that a player's current team = the team he played for in the last edition (IPL 2026).
#          That tells us whose retention list a player would appear on, and who RR can realistically buy.
# ─────────────────────────────────────────────────────────────────────
# 11a) Find the IPL 2026 competition and its matches.
ipl26_comp = comps[comps.comp_name == "IPL 2026"].comp_id.iloc[0]
m26 = matches[matches.comp_id == ipl26_comp][["match_id", "match_date"]]
team_names = teams[teams.comp_id == ipl26_comp].set_index("team_id").team_name

# 11b) FUNCTIONAL: `playing_eleven` lists the XI plus the impact-player substitute. `is_played = True` means
# the player actually took part (an impact sub who never came on has is_played = False).
played = xi[xi.match_id.isin(m26.match_id) & xi.is_played].merge(m26, on="match_id")

# 11c) For each player: how many IPL-2026 matches for each team, then pick the team with the MOST matches;
# ties are broken by the team he played for LAST (most recent match date).
# TECHNICAL: sort so the preferred row comes first, then drop_duplicates keeps that first row per player.
by_team = (played.groupby(["player_id", "team_id"])
                 .agg(matches_for_team=("match_id", "nunique"), last_match=("match_date", "max"))
                 .reset_index()
                 .sort_values(["player_id", "matches_for_team", "last_match"], ascending=[True, False, False]))
cur = by_team.drop_duplicates("player_id").copy()
cur["current_team"] = cur.team_id.map(team_names)
cur["current_team_source"] = "played IPL 2026"
movers = by_team.groupby("player_id").team_id.nunique()
print(f"Players who played for more than one franchise in IPL 2026 (mid-season move): {(movers > 1).sum()}")

# 11d) FALLBACK. Players in an IPL 2026 squad who never got a game (bench / injured) are still on that
# franchise's books, so we use the official squad list for them.
sq26 = squads[squads.comp_id == ipl26_comp][["player_id", "team_name"]].drop_duplicates("player_id")
fallback = sq26[~sq26.player_id.isin(cur.player_id)].rename(columns={"team_name": "current_team"})
fallback["current_team_source"] = "IPL 2026 squad, did not play"
cur = pd.concat([cur[["player_id", "current_team", "current_team_source", "matches_for_team"]],
                 fallback[["player_id", "current_team", "current_team_source"]]], ignore_index=True)
cur["matches_for_team"] = cur.matches_for_team.fillna(0).astype(int)

# 11e) CROSS-CHECK the two sources: for players found in both, does the squad team agree with the team he played for?
chk = cur[cur.current_team_source == "played IPL 2026"].merge(sq26.rename(columns={"team_name": "squad_team"}), on="player_id", how="left")
print(f"Played-for team agrees with the official squad list: {(chk.current_team == chk.squad_team).mean():.1%} "
      f"({(chk.squad_team.isna()).sum()} players not in any squad)")
display(chk[chk.current_team != chk.squad_team][["player_id", "current_team", "squad_team"]])

players = players.drop(columns=[c for c in ["current_team", "current_team_source", "matches_for_team"] if c in players.columns]
                       ).merge(cur, on="player_id", how="left")
players["current_team"] = players.current_team.fillna("Not in IPL 2026")
players["current_team_source"] = players.current_team_source.fillna("none")

display(players.groupby("current_team").agg(players=("player_id", "count"),
                                            played_2026=("current_team_source", lambda s: (s == "played IPL 2026").sum()),
                                            median_age=("age", "median")).sort_values("players", ascending=False))
rr = players[players.current_team == "Rajasthan Royals"].sort_values("matches_for_team", ascending=False)
print(f"\nRajasthan Royals players found: {len(rr)}")
display(rr[["player_id", "display_name", "age", "age_band", "matches_for_team", "current_team_source"]].head(30))
log_dq("team", "no 'current team' field in the data", (players.current_team != "Not in IPL 2026").sum(),
       "derived from IPL 2026 matches played (most matches, latest as tie-break); squad list as fallback")

Players who played for more than one franchise in IPL 2026 (mid-season move): 0
Played-for team agrees with the official squad list: 100.0% (0 players not in any squad)


,player_id,current_team,squad_team


,players,played_2026,median_age
current_team,,,
Not in IPL 2026,709,0,30.000
Chennai Super Kings,29,21,30.000
Gujarat Titans,26,19,30.000
Kolkata Knight Riders,26,19,28.500
Mumbai Indians,26,24,28.500
Delhi Capitals,25,22,28.000
Lucknow Super Giants,25,23,29.000
Rajasthan Royals,25,20,30.000
Sunrisers Hyderabad,25,19,28.000



Rajasthan Royals players found: 25


,player_id,display_name,age,age_band,matches_for_team,current_team_source
108,6819,Yashasvi Jaiswal,25.000,22-25,16.000,played IPL 2026
722,50284,Vaibhav Sooryavanshi,15.000,U22,16.000,played IPL 2026
418,3298,Jofra Archer,31.000,30-33,16.000,played IPL 2026
425,7260,Dhruv Jurel,25.000,22-25,16.000,played IPL 2026
491,8182,Donovan Ferreira,29.000,26-29,15.000,played IPL 2026
112,6423,Riyan Parag,25.000,22-25,14.000,played IPL 2026
71,152,Ravindra Jadeja,38.000,34+,14.000,played IPL 2026
857,67036,Brijesh Sharma,24.000,22-25,13.000,played IPL 2026
500,7312,Nandre Burger,31.000,30-33,13.000,played IPL 2026
173,7503,Ravi Bishnoi,26.000,26-29,9.000,played IPL 2026


[DQ] team | no 'current team' field in the data | 250 rows | derived from IPL 2026 matches played (most matches, latest as tie-break); squad list as fallback


### Cell 11: Current IPL team
**Procedure:** For each player, counted IPL 2026 matches by franchise (`is_played = True`) and picked the team with the most matches (latest match breaks ties). Players on an IPL 2026 squad who did not play were assigned from the official squad list. Cross-checked the two sources.

**Observations**
- No player played for two franchises in IPL 2026.
- The played-for team agrees with the official squad list for 100% of players, so the squad file is reliable.
- 250 players have a current IPL team, and 709 players (mostly international and A-team) are "Not in IPL 2026". These are the candidate pool for the auction. Players currently on another franchise's retention list will be removed once the official list is out on 17 Nov.
- Players per franchise: CSK 29, GT 26, KKR 26, MI 26, DC 25, LSG 25, RR 25, SRH 25, PBKS 22, RCB 21. RR has 25 on the books and 20 played.
- RR's core: Jaiswal, Jurel and Vaibhav Sooryavanshi (16 matches each), Archer (16), Parag (14), Jadeja (14, age 38). RR's older players (Jadeja 38, Shanaka 35, Milne 34) point to an ageing death/finisher group, which the later RR diagnostic can test.
- The squad list has 275 players, but only 250 appear in the ball data. The other 25 never batted or bowled in this dataset, so they cannot be rated. The squad table is saved anyway for slot counting in the auction layer.

In [13]:
# ─────────────────────────────────────────────────────────────────────
# CELL 12 · FINAL CHECKS AND SAVE
# Purpose: run a last set of sanity checks, then save the clean tables so notebook 02 starts from here
#          (no need to repeat cleaning), and save the data-quality log for the methodology report.
# ─────────────────────────────────────────────────────────────────────
# 12a) Final assertions. TECHNICAL: `assert` stops the notebook with an error if a rule is broken, which is
# better than silently carrying a bad table forward.
assert not ball.duplicated(["match_id", "innings", "innings_ball_num"]).any(), "duplicate deliveries"
assert ball.innings.isin([1, 2]).all(), "super overs leaked back in"
assert ball.groupby(["match_id", "innings"]).is_legal.sum().max() <= MAX_LEGAL_BALLS
assert ball.groupby(["match_id", "innings"]).is_wicket_clean.sum().max() <= 10
assert ball.bowl_speed_clean.between(50, 160).all()
assert players.player_id.is_unique
assert ball.striker_id.isin(players.player_id).all() and ball.bowler_id.isin(players.player_id).all()
print("All final checks passed.")

# 12b) Keep a tidy set of columns on the ball table, in a sensible order.
ball_cols = ["match_id", "innings", "overid", "over_ball_num", "innings_ball_num", "batting_team_id", "bowling_team_id",
             "striker_id", "non_striker_id", "bowler_id", "batting_type", "bowling_type", "pace_or_spin", "batting_order",
             "runs", "extras_type", "extras_runs", "total_runs", "is_legal", "is_extra", "is_four", "is_six",
             "is_wicket", "is_wicket_clean", "wicket_imputed", "possible_retired_hurt", "bowler_wicket",
             "dismissal_type", "dismissed_player_id", "bowl_speed", "bowl_speed_clean", "speed_imputed", "speed_source",
             "stroke", "delivery_style", "over_or_around", "on_or_off", "tier", "venue_id", "day_night"]
ball_out = ball[ball_cols]

# 12c) Save. TECHNICAL: parquet is a compact, fast, typed format but needs the `pyarrow` package. If it is not
# installed we fall back to pickle (Python's native format; also keeps types, just not shareable outside Python).
def save_table(df, name):
    try:
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False)
        return "parquet"
    except Exception:
        df.to_pickle(OUT_DIR / f"{name}.pkl")
        return "pickle"

saved = {name: (len(df), save_table(df, name)) for name, df in
         {"ball_clean": ball_out, "matches_clean": matches, "players": players, "venues": venues.reset_index(),
          "teams": teams, "squads": squads, "playing_eleven": xi}.items()}
dq = pd.DataFrame(dq_log)
dq.to_csv(OUT_DIR / "data_quality_log.csv", index=False)
display(pd.DataFrame(saved, index=["rows", "format"]).T)

print("\nDATA-QUALITY LOG (this goes into the methodology report):")
display(dq)

# 12d) Headline numbers to quote in the report.
print(f"\nClean analysis set: {ball_out.match_id.nunique()} matches, {len(ball_out):,} deliveries, {players.shape[0]} players")
print("Matches by tier:", matches[matches.n_innings > 0].tier.value_counts().to_dict())

All final checks passed.


,rows,format
ball_clean,200961,parquet
matches_clean,883,parquet
players,959,parquet
venues,98,parquet
teams,353,parquet
squads,6413,parquet
playing_eleven,28381,parquet



DATA-QUALITY LOG (this goes into the methodology report):


,step,issue,rows_affected,action
0,scope,super-over deliveries (innings 3-4) removed,38,dropped from main analysis
1,scope,matches with only one innings bowled,22,"kept, flagged complete_match=False"
2,runs,bat-only `runs` understates team runs,11826,extras1_runs (real extras) + overthrows -> tot...
3,wickets,dismissed player named but is_wicket False -> ...,11,"is_wicket_clean=True (type unknown, not credit..."
4,wickets,imputation undone to respect the 10-wicket rul...,1,"left as non-wicket, flagged"
5,speed,bowl_speed zero / negative / out-of-range,77908,replaced by bowler -> bowling-type -> pace/spi...
6,totals,rebuilt innings totals vs official margins (no...,805,run-margin within 3: 100%; exact 100%; wicket-...
7,age,players with no age in the age file,5,"kept, flagged age_missing; no imputation"
8,identity,player names shared by two player IDs,6,display_name carries '#id'; joins use player_id
9,time,timestamps mix UTC and IST clock conventions u...,148,IPL start normalised to IST; day_night derived...



Clean analysis set: 867 matches, 200,961 deliveries, 959 players
Matches by tier: {'T20I': 470, 'IPL': 367, 'A-team / emerging': 30}


### Cell 12: Final checks and save
**Procedure:** Asserted the core rules (no duplicate deliveries, no super overs, at most 120 legal balls and 10 wickets per innings, speeds within 50–160, unique player IDs, every batter and bowler present in the player table), kept an ordered set of columns, saved seven tables and the data-quality log.

**Observations**
- All final checks passed.
- Saved as parquet in `interim/`: `ball_clean` (200,961 rows), `matches_clean` (883), `players` (959), `venues` (98), `teams` (353), `squads` (6,413) and `playing_eleven` (28,381), plus `data_quality_log.csv`.
- Analysis set: 867 matches (IPL 367, T20I 470, A-team 30), 200,961 deliveries and 959 players.
- The log records 10 steps (38 super-over balls dropped; 11 wickets imputed and 1 undone; 11,826 extras added; 77,908 speeds repaired; 5 missing ages; 6 name clashes; IPL time and venue labelling; current team derived). It is the data-cleaning table for the methodology report.
- Notebook 02 starts from these files, so the cleaning is never repeated.